# OpenBB local — verificacion de las 8 familias de analisis

Todo corre contra la instalacion local (`C:\Users\DELL\.ipython\OpenBB`), **sin claves de API**.

Antes de abrir este notebook, arranca la API:

```powershell
cd C:\Users\DELL\.ipython\OpenBB
.\run-openbb-api.ps1
```

Las series se guardan en `data/cache/*.parquet` para no repetir descargas.

**Como se lee esta tabla:** 

| Familia | Estado | Fuente |
|---|---|---|
| 1 Tecnico | completo | CBOE (indices) + TMX (OHLC) + calculo local |
| 2 Fundamental | completo | SEC EDGAR |
| 3 Macro | completo | Reserva Federal, Tesoro, OECD, BCE |
| 4 Sentimiento | completo | CFTC (COT), FINRA (shorts), Deribit (funding) |
| 5 Order flow | parcial | Deribit (cripto). Acciones nivel 2: requiere pago |
| 6 On-chain | hueco | no hay provider on-chain instalado |
| 7 Cuantitativo | completo | Fama-French + quantitative + econometrics |
| 8 Conductual | parcial | estacionalidad calculada del historico |

In [ ]:
# Entorno: resuelve SSL (BCE), desactiva auto-build y expone la cache
import sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "scripts" / "obb_env.py").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from obb_env import obb_local, consultar, CACHE_DIR

obb = obb_local()
print("cache:", CACHE_DIR)
print("comandos disponibles:", len(obb.coverage.commands))

## 1. Tecnico — *¿donde entro y salgo?*

In [ ]:
spx = consultar("spx", lambda: obb.index.price.historical("SPX", provider="cboe"))
serie = spx.reset_index()

rsi = consultar("spx_rsi", lambda: obb.technical.rsi(data=serie, index="date", target="close"))
ema = consultar("spx_ema20", lambda: obb.technical.ema(data=serie, index="date", target="close", length=20))
macd = consultar("spx_macd", lambda: obb.technical.macd(data=serie, index="date", target="close"))

print("SPX:", spx.shape, "| RSI:", rsi.shape, "| EMA20:", ema.shape, "| MACD:", macd.shape)
rsi.tail(3)

In [ ]:
# Indicadores que necesitan maximo/minimo: TMX es la unica fuente OHLC sin clave
shop = consultar("shop_ohlc", lambda: obb.equity.price.historical("SHOP", provider="tmx"))
e = shop.reset_index()

atr = consultar("shop_atr", lambda: obb.technical.atr(data=e, index="date", high="high", low="low", close="close"))
vol = consultar("shop_vol", lambda: obb.technical.realized_volatility(data=e, index="date", high="high", low="low", close="close"))

print("OHLC:", shop.shape, "| ATR:", atr.shape, "| Vol realizada:", vol.shape)
atr.tail(3)

## 2. Fundamental — *¿vale lo que cuesta?*

In [ ]:
bal = consultar("aapl_balance", lambda: obb.equity.fundamental.balance("AAPL", provider="sec", limit=4))
inc = consultar("aapl_income", lambda: obb.equity.fundamental.income("AAPL", provider="sec", limit=4))

print("Balance:", bal.shape, "| Income:", inc.shape)
inc[["period_ending", "total_revenue", "net_income"]].to_string()

## 3. Macro / intermercado — *¿el entorno permite riesgo?*

In [ ]:
effr = consultar("effr", lambda: obb.fixedincome.rate.effr(provider="federal_reserve"))
curva = consultar("yield_curve", lambda: obb.fixedincome.government.yield_curve(provider="federal_reserve"))
cpi = consultar("cpi", lambda: obb.economy.cpi(provider="oecd"))
pib = consultar("gdp", lambda: obb.economy.gdp.real(provider="oecd"))
eurusd = consultar("eurusd", lambda: obb.currency.price.historical("EURUSD", provider="ecb"))

print("EFFR:", effr.shape, "| Curva:", curva.shape, "| CPI:", cpi.shape, "| PIB:", pib.shape, "| EURUSD:", eurusd.shape)
curva.tail(4)

## 4. Sentimiento / posicionamiento — *¿que esta descontando la masa?*

In [ ]:
cot = consultar("cot_positioning", lambda: obb.cftc.cot_positioning(provider="cftc"))
shorts = consultar("shorts", lambda: obb.equity.shorts.short_interest(symbol="AAPL"))
funding = consultar("funding", lambda: obb.deribit.rates.funding_history(provider="deribit"))

print("COT:", cot.shape, "| Short interest:", shorts.shape, "| Funding:", funding.shape)
funding.tail(3)

## 5. Order flow — *¿quien esta comprando ahora?*

Libro de ordenes real (Deribit). Para acciones, el nivel 2 es de pago.

In [ ]:
libro = consultar("order_book", lambda: obb.deribit.market.order_book(symbol="BTC-PERPETUAL", provider="deribit"))
ticker = consultar("ticker", lambda: obb.deribit.market.ticker(symbol="BTC-PERPETUAL", provider="deribit"))
trades = consultar("trades", lambda: obb.deribit.market.trades(provider="deribit"))

print("Libro:", libro.shape, "| Ticker:", ticker.shape, "| Trades:", trades.shape)
libro.head(6)

## 6. On-chain — *¿que hacen los tenedores reales?*

**Hueco confirmado:** no hay provider on-chain instalado (MVRV, netflow, SOPR).
Para cubrirlo haria falta un provider adicional (Glassnode/CryptoQuant/Dune).

## 7. Cuantitativo — *¿tiene ventaja comprobable?*

In [ ]:
ff = consultar("factores", lambda: obb.famafrench.factors(provider="famafrench"))
resumen = consultar("stats_spx", lambda: obb.quantitative.summary(data=spx.reset_index(), index="date", target="close"))
ols = consultar("ols", lambda: obb.econometrics.ols_regression(
    data=ff.reset_index(), y_column="mkt_rf", x_columns=["smb", "hml"]))

print("Factores:", ff.shape, "| Resumen:", resumen.shape)
resumen

## 8. Conductual / estacionalidad — *¿que patrones de calendario existen?*

Calculado sobre 13.057 sesiones del S&P 500.

In [ ]:
import pandas as pd

s = spx.copy()
s.index = pd.to_datetime(s.index)
mensual = s["close"].resample("ME").last().pct_change().dropna() * 100

tabla = mensual.groupby(mensual.index.month).agg(
    rendimiento_medio="mean", volatilidad="std", anios="count").round(2)
tabla.index = [pd.Timestamp(2024, m, 1).strftime("%b") for m in tabla.index]
tabla.sort_values("rendimiento_medio", ascending=False)

## Conclusion

OpenBB local cubre **6 de las 8 familias** sin gastar un centavo en datos.

| Familia | Estado |
|---|---|
| Tecnico | cubierto |
| Fundamental | cubierto |
| Macro | cubierto |
| Sentimiento | cubierto |
| Order flow | parcial (cripto si, acciones no) |
| On-chain | **hueco** |
| Cuantitativo | cubierto |
| Conductual | parcial |

**El error a evitar:** usar las 8 como comite de votacion. 
Un filtro de contexto (macro o fundamental) + un gatillo tecnico + una validacion cuantitativa.

**Siguiente paso:** exportar estas series como *informative pairs* de Freqtrade,
para que la estrategia tecnica decida con el contexto macro al lado.